# SEM SimCLR / BYOL — Colab pipeline

Обучение contrastive моделей на SEM-тайлах с полным resume-safe чекпоинтом,
TensorBoard-логированием в Drive, unified evaluation.

**Подготовка:** `Среда выполнения` → `Сменить среду выполнения` → T4 / A100 / V100.

**Синхронизация с кодовой базой:**
- Checkpoint format: полный dict (model/optimizer/scheduler/val_loss) из PR #2.
- Projection head: v2 (с BatchNorm), унифицировано с Crystal (PR #3).
- BYOL tau: cosine ramp-up `base → 1.0` (PR #2).
- Seeds: зафиксированы глобально через `src.utils.repro.set_global_seed(42)` (PR #1).
- Augmentations: параметризованы через CLI (`--crop_scale_min`, `--blur_p`, `--noise_std` и т.д., PR #8).
- Validation split: опциональный (`--val_frac 0.1`), best-checkpoint по val_loss (PR #7).
- TensorBoard: inline-viewer в Colab (PR #7).
- Unified evaluation: `run_sem_evaluation.py` выдаёт markdown-отчёт с bootstrap CI (PR #7).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# === Код из GitHub + данные из Drive + зависимости ===
import os

# 1. Клонируем / обновляем код
if os.path.exists('/content/diploma_project/.git'):
    !cd /content/diploma_project && git pull
else:
    !git clone https://github.com/daniilctrl/sem-image-analysis.git /content/diploma_project

# 2. Распаковываем данные из zip (только если ещё не распакованы)
if not os.path.exists('/content/diploma_project/data/processed/tiles_metadata.csv'):
    !cp "/content/drive/MyDrive/diploma_data/processed_data_v2.zip" /content/
    !unzip -qo /content/processed_data_v2.zip -d /content/_data_tmp/
    !mkdir -p /content/diploma_project/data/processed
    !cp -r /content/_data_tmp/* /content/diploma_project/data/processed/
    !rm -rf /content/_data_tmp /content/processed_data_v2.zip
    print(f"Data extracted: {len(os.listdir('/content/diploma_project/data/processed'))} files")
else:
    print('Data already in place.')

# 3. Установка проекта в editable режиме с extras (viz + dev для tensorboard)
%cd /content/diploma_project
!pip install -q -e ".[viz,dev]"

# Быстрая проверка: pyproject работает, smoke-тесты парсятся.
!python -c "from src.utils.repro import set_global_seed; set_global_seed(42); print('repro OK')"
!python -c "from src.models.deep_clustering.augmentations import AugmentationConfig; print('aug config:', AugmentationConfig().to_dict())"

---
## 3. Обучение

Выбери одну из конфигураций:

- **3a** — SimCLR с нуля, default aug, `τ=0.5`, `val_frac=0.1`
- **3b** — SimCLR resume после обрыва сессии Colab
- **3c** — SimCLR alternative (`τ=0.2` + strong crop, для ablation)
- **3d** — TensorBoard live-viewer (запускать параллельно с обучением)
- **4** — BYOL с cosine tau ramp-up

In [ ]:
# === 3a: SimCLR — первый запуск (full resume-safe pipeline) ===
# Полный формат чекпоинта: сохраняются model + optimizer + scheduler + val_loss.
# TensorBoard-логи на Drive (можно открыть ниже в ячейке 3d).
# Val split 10% — best-checkpoint будет выбран по val_loss, не train avg.

RUN_NAME = "simclr_v2_t05_default_aug"

!python /content/diploma_project/src/models/deep_clustering/train.py \
    --data_dir /content/diploma_project/data/processed \
    --metadata_path /content/diploma_project/data/processed/tiles_metadata.csv \
    --output_dir /content/drive/MyDrive/diploma_checkpoints/{RUN_NAME} \
    --epochs 30 \
    --batch_size 64 \
    --learning_rate 3e-4 \
    --temperature 0.5 \
    --workers 4 \
    --val_frac 0.1 \
    --save_every 5 \
    --seed 42 \
    --tb_log_dir /content/drive/MyDrive/diploma_logs/{RUN_NAME}

In [ ]:
# === 3b: SimCLR — resume после обрыва Colab-сессии ===
# Благодаря полному формату чекпоинта optimizer/scheduler восстанавливаются
# в точности — первые шаги после resume идут без потери momentum.

RUN_NAME = "simclr_v2_t05_default_aug"
LAST_EPOCH = 15           # <-- номер последнего завершённого чекпоинта
TOTAL_EPOCHS = 30
REMAINING = TOTAL_EPOCHS - LAST_EPOCH

!python /content/diploma_project/src/models/deep_clustering/train.py \
    --data_dir /content/diploma_project/data/processed \
    --metadata_path /content/diploma_project/data/processed/tiles_metadata.csv \
    --output_dir /content/drive/MyDrive/diploma_checkpoints/{RUN_NAME} \
    --epochs {REMAINING} \
    --batch_size 64 \
    --workers 4 \
    --val_frac 0.1 \
    --seed 42 \
    --tb_log_dir /content/drive/MyDrive/diploma_logs/{RUN_NAME} \
    --resume /content/drive/MyDrive/diploma_checkpoints/{RUN_NAME}/simclr_resnet50_epoch_{LAST_EPOCH}.pth \
    --start_epoch {LAST_EPOCH}

In [ ]:
# === 3c: SimCLR — альтернативная конфигурация (ablation) ===
# Пример, как менять аугментации и hyperparameters через CLI без правки кода.
# Здесь: более низкий temperature + более агрессивный RandomResizedCrop.

RUN_NAME = "simclr_v2_t02_strong_crop"

!python /content/diploma_project/src/models/deep_clustering/train.py \
    --data_dir /content/diploma_project/data/processed \
    --metadata_path /content/diploma_project/data/processed/tiles_metadata.csv \
    --output_dir /content/drive/MyDrive/diploma_checkpoints/{RUN_NAME} \
    --epochs 30 \
    --batch_size 96 \
    --learning_rate 3e-4 \
    --temperature 0.2 \
    --workers 4 \
    --val_frac 0.1 \
    --save_every 5 \
    --seed 42 \
    --tb_log_dir /content/drive/MyDrive/diploma_logs/{RUN_NAME} \
    --crop_scale_min 0.08 \
    --blur_p 0.5 \
    --noise_std 0.04

In [ ]:
# === 4: BYOL с cosine tau ramp-up ===
# tau растёт от 0.996 до 1.0 по косинусу за весь прогон (Grill et al. 2020).
# best-checkpoint по val_loss благодаря --val_frac.

RUN_NAME = "byol_cosine_30ep"

!python /content/diploma_project/src/models/deep_clustering/train_byol.py \
    --data_dir /content/diploma_project/data/processed \
    --metadata_path /content/diploma_project/data/processed/tiles_metadata.csv \
    --output_dir /content/drive/MyDrive/diploma_checkpoints/{RUN_NAME} \
    --epochs 30 \
    --batch_size 64 \
    --learning_rate 3e-4 \
    --ema_tau 0.996 \
    --ema_tau_schedule cosine \
    --workers 4 \
    --val_frac 0.1 \
    --save_every 5 \
    --seed 42 \
    --tb_log_dir /content/drive/MyDrive/diploma_logs/{RUN_NAME}

In [ ]:
# === 3d: TensorBoard — live-мониторинг training curves ===
# Посмотреть curves (train/loss_step, val/loss_epoch, grad_norm, tau) прямо в Colab.
# Запускать ПОСЛЕ старта обучения или после нескольких сохранённых логов.

%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/diploma_logs/

---
## 5. Извлечение эмбеддингов

После обучения SimCLR и/или BYOL извлекаем frozen-эмбеддинги для всех тайлов.
Сохраняется в `data/embeddings/{simclr,byol}/finetuned_embeddings.npy` + companion `.csv` с tile_names.

Чекпоинт может быть в любом из форматов (авто-detect):
- новый dict с `model_state_dict`
- legacy raw `state_dict`

Версия projection head (v1/v2) определяется автоматически по BN-ключам.

In [ ]:
# === 5: Извлечение эмбеддингов из best-checkpoint ===
# Поддерживает оба формата чекпоинта: новый dict с model_state_dict
# и legacy raw state_dict (auto-detect в SimCLR.from_state_dict).
# Версия projection head определяется по BN-ключам автоматически.

RUN_NAME = "simclr_v2_t05_default_aug"
CHECKPOINT = f'/content/drive/MyDrive/diploma_checkpoints/{RUN_NAME}/simclr_resnet50_best.pth'
MODEL_TYPE = 'simclr'  # 'simclr' или 'byol'
OUTPUT_SUBDIR = 'simclr'  # эмбеддинги SimCLR пойдут в data/embeddings/simclr/

# Для BYOL:
# RUN_NAME = "byol_cosine_30ep"
# CHECKPOINT = f'/content/drive/MyDrive/diploma_checkpoints/{RUN_NAME}/byol_resnet50_best.pth'
# MODEL_TYPE = 'byol'
# OUTPUT_SUBDIR = 'byol'

!python /content/diploma_project/src/models/deep_clustering/extract_simclr_embeddings.py \
    --checkpoint {CHECKPOINT} \
    --model_type {MODEL_TYPE} \
    --data_dir /content/diploma_project/data/processed \
    --metadata_path /content/diploma_project/data/processed/tiles_metadata.csv \
    --output_dir /content/diploma_project/data/embeddings/{OUTPUT_SUBDIR} \
    --batch_size 128 \
    --workers 4 \
    --seed 42

In [ ]:
# === 5b: Опциональная UMAP-визуализация (baseline vs fine-tuned) ===
# extract_simclr_embeddings.py генерирует PNG как побочный артефакт.
# Этот сам ячейка — просто показать его в Colab.

from IPython.display import Image, display
import os

for subdir in ['simclr', 'byol']:
    umap_path = f'/content/diploma_project/data/embeddings/{subdir}/baseline_vs_{subdir}_umap.png'
    if os.path.exists(umap_path):
        print(f"\nUMAP: Baseline vs {subdir.upper()}")
        display(Image(umap_path))

---
## 6. Unified SEM Evaluation

Единый pipeline, который делает 4 оценки и собирает их в один markdown-отчёт:

1. **Cross-Scale Retrieval** — P@K material + P@K cross-scale с bootstrap 95% CI
2. **Scale Invariance Metrics** — Cramér's V, AMI, MagVarRatio, Entropy (adaptive mag bins)
3. **Linear Probe** — stratified K-fold LR на `sft_annotations.csv` с CI
4. **k-NN Probe** — weighted cosine k-NN с K ∈ {1, 5, 20} с CI

Все результаты → `data/results/sem_eval_report_<timestamp>.md` (полный markdown)
+ отдельные CSV-артефакты для каждой секции.

In [ ]:
# === 6a: Sanity check — все эмбеддинги на месте ===
import os

EMB_DIR = '/content/diploma_project/data/embeddings'

expected = [
    ('Baseline ResNet50', 'resnet50_embeddings.npy', 'embedding_names.csv'),
    ('SimCLR',            'simclr/finetuned_embeddings.npy', 'simclr/finetuned_embedding_names.csv'),
    ('BYOL',              'byol/finetuned_embeddings.npy', 'byol/finetuned_embedding_names.csv'),
]
for name, emb_rel, names_rel in expected:
    emb = os.path.join(EMB_DIR, emb_rel)
    names = os.path.join(EMB_DIR, names_rel)
    emb_ok = os.path.exists(emb)
    names_ok = os.path.exists(names)
    status = "OK" if (emb_ok and names_ok) else "MISSING"
    print(f"  [{status}] {name}: emb={emb_ok}, names={names_ok}")

# Если embedding_names.csv устарел — регенерация в один клик:
# !python /content/diploma_project/scripts/regenerate_embedding_names.py --all

In [ ]:
# === 6b: Unified SEM Evaluation — четыре оценки + markdown-отчёт ===
# Запускает Cross-Scale Retrieval, Scale Invariance, Linear Probe, k-NN Probe.
# Все интервальные метрики снабжены bootstrap 95% CI.
# Итог: data/results/sem_eval_report_<timestamp>.md + отдельные CSV.

%cd /content/diploma_project

!python src/evaluation/run_sem_evaluation.py \
    --meta_path data/processed/tiles_metadata.csv \
    --emb_dir data/embeddings \
    --annotations_path data/sft_annotations.csv \
    --output_dir /content/drive/MyDrive/diploma_results/ \
    --K 10 \
    --n_clusters 4 \
    --n_folds 5 \
    --knn_k 1 5 20 \
    --seed 42

In [ ]:
# === 6c: Показать markdown-отчёт прямо в ноутбуке ===
from IPython.display import display, Markdown
import glob, os

RESULTS = '/content/drive/MyDrive/diploma_results'

reports = sorted(glob.glob(f'{RESULTS}/sem_eval_report_*.md'))
if reports:
    latest = reports[-1]
    print(f"Latest report: {latest}\n")
    with open(latest, 'r', encoding='utf-8') as f:
        display(Markdown(f.read()))
else:
    print(f"No markdown reports in {RESULTS}. Run the evaluation cell above first.")

In [ ]:
# === 6d: Артефакты на Drive — полный список для сдачи ===
import glob, os

def list_dir(path, label):
    print(f"\n{label}  ({path}):")
    for f in sorted(glob.glob(f'{path}/**/*', recursive=True)):
        if os.path.isfile(f):
            size_mb = os.path.getsize(f) / (1024 * 1024)
            rel = os.path.relpath(f, path)
            suffix = f"{size_mb:8.2f} MB" if size_mb >= 0.1 else f"{size_mb*1024:8.1f} KB"
            print(f"  {rel:60s} {suffix}")

list_dir('/content/drive/MyDrive/diploma_results', 'Results (CSV + markdown)')
list_dir('/content/drive/MyDrive/diploma_checkpoints', 'Checkpoints (.pth)')
list_dir('/content/drive/MyDrive/diploma_logs', 'TensorBoard logs')

print("\nDone — all artifacts saved to Google Drive.")